# NB03 · Semana 7 — DuckDB y Delta sobre el lake

**Ingeniería de Datos — LCD · UCA Rosario · 2026**  
Versión: **para el alumno (a completar)**

## Objetivo

Consultar y manipular datos del lake con **DuckDB** sobre **Delta Lake**: leer la tabla bronze, ejecutar consultas analíticas, y usar *time-travel* para leer versiones anteriores.

## 1. Leer la tabla Delta bronze

Usamos la librería `deltalake` para cargar la tabla a un DataFrame, y luego DuckDB para consultarla con SQL.

In [1]:
from deltalake import DeltaTable
import duckdb, os

STORAGE_OPTIONS = {
    'AWS_ACCESS_KEY_ID': os.environ.get('AWS_ACCESS_KEY_ID', 'minioadmin'),
    'AWS_SECRET_ACCESS_KEY': os.environ.get('AWS_SECRET_ACCESS_KEY', 'minioadmin'),
    'AWS_ENDPOINT_URL': os.environ.get('AWS_ENDPOINT_URL', 'http://minio:9000'),
    'AWS_ALLOW_HTTP': 'true',
}
BUCKET = os.environ.get("MINIO_BUCKET", "lake")
BRONZE = f"s3://{BUCKET}/bronze/series"
# TODO(alumno): cargar la tabla Delta a un DataFrame 'df'
# Cargar la tabla Delta a un DataFrame 'df'
dt = DeltaTable(BRONZE, storage_options=STORAGE_OPTIONS)
df = dt.to_pandas()

print(f"Conectado a: {BRONZE}")
print("Filas y columnas en Bronze:", df.shape)

Conectado a: s3://grupo07/bronze/series
Filas y columnas en Bronze: (73434, 9)


## 2. Consultas analíticas con DuckDB

DuckDB consulta el DataFrame directamente con SQL, sin cargarlo a ninguna base.

In [2]:
# TODO(alumno): con duckdb.sql(...), calcular por idVariable:
#   cantidad de observaciones, promedio, mínimo y máximo del valor.
# --- Exploración Analítica con DuckDB ---

print("1. Resumen por Tipo de Amenaza y Estado Actual")
# Agrupamos por threat y vemos cuántas siguen vivas ('online')
resumen_amenazas = duckdb.sql("""
    SELECT 
        threat,
        COUNT(*) as total_urls,
        SUM(CASE WHEN url_status = 'online' THEN 1 ELSE 0 END) as urls_activas,
        ROUND(SUM(CASE WHEN url_status = 'online' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as porcentaje_activas
    FROM df
    GROUP BY threat
    ORDER BY total_urls DESC
""").df()
print(resumen_amenazas.head(5))


print("\n2. Top 5 Reporteros Históricos")
# Identificamos qué usuarios o entidades alimentan más la base de datos
top_reporteros = duckdb.sql("""
    SELECT 
        reporter,
        COUNT(*) as amenazas_reportadas,
        MIN(CAST(date_added AS DATE)) as primer_reporte,
        MAX(CAST(date_added AS DATE)) as ultimo_reporte
    FROM df
    WHERE reporter IS NOT NULL
    GROUP BY reporter
    ORDER BY amenazas_reportadas DESC
    LIMIT 5
""").df()
print(top_reporteros)


print("\n3. Tendencia de Detección (Últimos 5 días con datos)")
# Agrupamos por fecha pura para ver el volumen de inyección diario
tendencia_diaria = duckdb.sql("""
    SELECT 
        CAST(date_added AS DATE) as fecha,
        COUNT(*) as nuevas_amenazas
    FROM df
    GROUP BY fecha
    ORDER BY fecha DESC
    LIMIT 5
""").df()
print(tendencia_diaria)

1. Resumen por Tipo de Amenaza y Estado Actual
             threat  total_urls  urls_activas  porcentaje_activas
0  malware_download       73434       27998.0               38.13

2. Top 5 Reporteros Históricos
       reporter  amenazas_reportadas primer_reporte ultimo_reporte
0        shmulc                15227     2026-10-06     2026-10-18
1     adliwahid                 8999     2026-05-15     2026-10-20
2      geenensp                 8601     2023-10-02     2026-10-20
3  cesnet_certs                 6680     2024-08-30     2026-10-20
4     anonymous                 5867     2018-08-02     2026-10-20

3. Tendencia de Detección (Últimos 5 días con datos)
       fecha  nuevas_amenazas
0 2026-10-20              624
1 2026-10-19              390
2 2026-10-18              520
3 2026-10-17              453
4 2026-10-16              379


## 3. Time-travel sobre Delta

Delta guarda el historial de versiones. Podemos leer una versión anterior de la tabla (por ejemplo, el estado tras la primera ingesta).

⚠️ **Requiere el lake activo con la tabla bronze poblada en más de una versión** (ver NB02).

In [3]:
# TODO(alumno): listar las versiones del historial Delta y leer la version=0.
# dt = DeltaTable(BRONZE, storage_options=STORAGE_OPTIONS)
# versiones = ...
# df_v0 = ...

# con duckdb.sql(...), calcular por threat
# cantidad de observaciones, primera y última detección.
resumen = duckdb.sql("""
    SELECT 
        threat,
        COUNT(*) as cantidad_urls,
        MIN(date_added) as primera_deteccion,
        MAX(date_added) as ultima_deteccion
    FROM df
    GROUP BY threat
    ORDER BY cantidad_urls DESC
""").df()

print(resumen)

             threat  cantidad_urls         primera_deteccion  \
0  malware_download          73434 2018-06-05 22:09:14+00:00   

           ultima_deteccion  
0 2026-10-20 20:31:17+00:00  


## Cierre

DuckDB + Delta permiten trabajar el lake como si fuera una base analítica, con la ventaja del versionado. En la próxima clase formalizamos las transformaciones con **dbt**.